In [1]:
from Crypto.Cipher import AES
from Crypto.Util.Padding import unpad
import hashlib

def inv_mod(x, p):
    return pow(x, p - 2, p)

def ecc_points_add(P, Q, a, p):
    if P == "Origin":
        return Q
    if Q == "Origin":
        return P
    if P[0] == Q[0] and (P[1] + Q[1]) % p == 0:
        return "Origin"
    if P != Q:
        lam = ((Q[1] - P[1]) * inv_mod(Q[0] - P[0], p)) % p
    else:
        lam = ((3 * pow(P[0], 2) + a) * inv_mod(2 * P[1], p)) % p
    x3 = (pow(lam, 2) - P[0] - Q[0]) % p
    y3 = (lam * (P[0] - x3) - P[1]) % p
    return (x3, y3)

def scalar_mul(P, n, a, p):
    R = "Origin"
    Q = P
    while n > 0:
        if n % 2 == 1:
            R = ecc_points_add(R, Q, a, p)
        Q = ecc_points_add(Q, Q, a, p)
        n //= 2
    return R

def decrypt_flag(shared_secret: int, iv: str, ciphertext: str):
    sha1 = hashlib.sha1()
    sha1.update(str(shared_secret).encode('ascii'))
    key = sha1.digest()[:16]
    cipher = AES.new(key, AES.MODE_CBC, bytes.fromhex(iv))
    plaintext = cipher.decrypt(bytes.fromhex(ciphertext))
    return unpad(plaintext, 16).decode('ascii')

def solve():
    a = 497
    b = 1768
    p = 9739
    x_QA = 4726
    n_B = 6534
    
    iv = "cd9da9f1c60925922377ea952afc212c"
    encrypted_flag = "febcbe3a3414a730b125931dccf912d2239f3e969c4334d95ed0ec86f6449ad8"
    
    y_squared = (pow(x_QA, 3) + a * x_QA + b) % p
    y1 = pow(y_squared, (p + 1) // 4, p)
    y2 = p - y1
    
    for y_QA in (y1, y2):
        if (pow(y_QA, 2) % p) == y_squared:
            Q_A = (x_QA, y_QA)
            shared_secret_point = scalar_mul(Q_A, n_B, a, p)
            try:
                flag = decrypt_flag(shared_secret_point[0], iv, encrypted_flag)
                print(flag)
                break
            except Exception:
                continue

if __name__ == "__main__":
    solve()

crypto{3ff1c1ent_k3y_3xch4ng3}
